In [ ]:
# !mkdir -p 'data/'
# !wget --user-agent "Mozilla" "https://arxiv.org/pdf/2307.09288.pdf" -O "data/llama2.pdf"

In [1]:
from pathlib import Path

from llama_index.readers.file import PDFReader
from llama_index.readers.file import PyMuPDFReader

from llama_index.core import (

    Settings,

)

In [2]:
md_files = list(Path("core/rag/parser/output").rglob("*.md")) if Path("core/rag/parser/output").exists() else []

In [3]:
from llama_index.core import SimpleDirectoryReader
docs = SimpleDirectoryReader(
input_files=[str(f) for f in md_files]
).load_data()

In [4]:
from llama_index.core.node_parser import (
    HierarchicalNodeParser,
    SentenceSplitter,
)

In [5]:
node_parser = HierarchicalNodeParser.from_defaults()

In [6]:
nodes = node_parser.get_nodes_from_documents(docs)

In [7]:
nodes

[TextNode(id_='786feeb7-126a-4165-80bd-90aa54dcd4f2', embedding=None, metadata={'file_path': 'core/rag/parser/output/group_bookings.md', 'file_name': 'group_bookings.md', 'file_type': 'text/markdown', 'file_size': 2365, 'creation_date': '2026-04-23', 'last_modified_date': '2026-04-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ce9cbd1a-2383-4e96-9e7d-4a8d71c5082a', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'core/rag/parser/output/group_bookings.md', 'file_name': 'group_bookings.md', 'file_type': 'text/markdown', 'file_size': 2365, 'creation_date': '2026-04-23', 'last_modified_date': '2026-04-24'}, hash='758584abe08a1eca1f0a0294b7bc01a64ce279bd27663209e2c5f4bfbae0301b'), <NodeRelati

In [8]:
from llama_index.core.node_parser import get_leaf_nodes, get_root_nodes

In [9]:
leaf_nodes = get_leaf_nodes(nodes)

In [10]:
root_nodes = get_root_nodes(nodes)

In [11]:
# define storage context
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import StorageContext
from llama_index.llms.ollama import Ollama
docstore = SimpleDocumentStore()

# insert nodes into docstore
docstore.add_documents(nodes)

# define storage context (will include vector store by default too)
storage_context = StorageContext.from_defaults(docstore=docstore)

Settings.llm = Ollama(
                model="phi4:latest",

                base_url="https://aerollm.share.zrok.io",
        
                request_timeout=120,
            )

/Users/Rajan/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
## Load index into vector index
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.ollama import OllamaEmbedding

embed_model = OllamaEmbedding(
                model_name="embeddinggemma",
                base_url="https://aerollm.share.zrok.io",
            )
base_index = VectorStoreIndex(
    leaf_nodes,
    storage_context=storage_context,
    embed_model=embed_model
)

2026-04-24 18:42:57,990 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:42:58,536 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:42:58,990 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:42:59,342 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:42:59,733 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:43:00,249 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:43:00,759 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:43:01,009 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:43:01,474 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24

In [13]:
from llama_index.core.retrievers import AutoMergingRetriever

base_retriever = base_index.as_retriever(similarity_top_k=6)
retriever = AutoMergingRetriever(base_retriever, storage_context, verbose=True)

In [26]:
# query_str = "What were some lessons learned from red-teaming?"
# query_str = "Can you tell me about the key concepts for safety finetuning"
query_str = (
"List all the prices for all the party types"
)

nodes = retriever.retrieve(query_str)
base_nodes = base_retriever.retrieve(query_str)

2026-04-24 18:46:55,977 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:46:56,489 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"


In [27]:
len(nodes)

6

In [28]:
from llama_index.core.response.notebook_utils import display_source_node

for node in nodes:
    display_source_node(node, source_length=10000)

**Node ID:** de46ed23-91cb-46e6-9349-d9407cce44b7<br>**Similarity:** 0.6115563952474332<br>**Text:** Socks are 50% off ($2.00 instead of $3.99). Go karting is NOT included but can be added on for $9.90 per person. No physical invitations included, and outside food requires a $50 fee. Total party duration is approximately 2.5 hours (90 min jump + 60 min room).

## Ultimate Party

The Ultimate Party is $59.90 per jumper (minimum 10 jumpers, base price $599).<br>

**Node ID:** 13ef96b6-6362-4e34-8cc2-a572e560b6e6<br>**Similarity:** 0.6114204658700021<br>**Text:** Juice Boxes (10 boxes) for $30.00. Soft Drink Cans (10 cans) for $20.00. Physical Birthday Invitations for $0.75 each. Outside Food Fee is $50.00 (waived for Ultimate Party). Go Kart Race add-on is $9.90 per person (already included in Ultimate Party). Late Ordering Fee (within 24 hours) is $20.00.<br>

**Node ID:** 264cc621-26c2-405b-8e83-df479ad3fe08<br>**Similarity:** 0.5795786469612371<br>**Text:** # Birthday Parties

## Premium Party

The Premium Party is $39.90 per jumper (minimum 10 jumpers, base price $399). Includes 75 minutes on the trampoline courts followed by 45 minutes in a private party room (fits 25-30 people). Food included: 1 large pizza per 5 jumpers and 1 juice box per jumper. The birthday child gets a FREE glow t-shirt, digital invitations, and a FREE 60-minute jump pass.<br>

**Node ID:** f6dc3e11-2a91-4244-885f-246fc9d7d95c<br>**Similarity:** 0.572270361656494<br>**Text:** And the Ultimate Party is $59.90 per jumper with 120 minutes of jump time, 60 minutes of room time, plus go karting for every jumper. All packages include pizza and drinks. Minimum 10 jumpers. Would you like more details on any package?

*Note: Pause between packages for clarity*

## Gokart Overview

> We have go karting on two tracks.<br>

**Node ID:** afceebfa-5d79-4f06-833e-841043bbb98a<br>**Similarity:** 0.5721816090507775<br>**Text:** ## All Packages

AeroSports Scarborough offers 3 birthday party packages: Premium ($39.90/jumper), VIP ($44.90/jumper), and Ultimate ($59.90/jumper). All require a minimum of 10 jumpers. All include pizza, drinks, a private party room, a glow t-shirt, digital invitations, and a FREE 60-min add-on pass for guest jumpers.<br>

**Node ID:** 97301de6-318a-47d1-ade4-28c9180e177b<br>**Similarity:** 0.5720488250455483<br>**Text:** Cost difference is $15 per jumper ($150 more for 10 jumpers).

**Best Value:** The Ultimate Party offers the most value with go karting alone worth $9.90/jumper ($99 for 10 kids), plus free socks ($3.99/jumper = $39.90 for 10), and the waived $50 outside food fee. That is roughly $189 in extras for $150 more than VIP.<br>

In [17]:
from llama_index.core.query_engine import RetrieverQueryEngine

In [18]:
query_engine = RetrieverQueryEngine.from_args(retriever)
base_query_engine = RetrieverQueryEngine.from_args(base_retriever)

2026-04-24 18:44:22,998 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/show "HTTP/1.1 200 OK"


In [19]:
response = query_engine.query(query_str)

2026-04-24 18:44:24,625 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:44:24,636 - INFO - > Merging 3 nodes into parent node.
> Parent node id: a727ac38-b557-4ddf-b157-f3f72c50487f.
> Parent node text: # Birthday Parties

## Premium Party

The Premium Party is $39.90 per jumper (minimum 10 jumpers,...



> Merging 3 nodes into parent node.
> Parent node id: a727ac38-b557-4ddf-b157-f3f72c50487f.
> Parent node text: # Birthday Parties

## Premium Party

The Premium Party is $39.90 per jumper (minimum 10 jumpers,...



2026-04-24 18:44:25,850 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/chat "HTTP/1.1 200 OK"


In [20]:
response.response

"The Ultimate Party is the package that includes go karting as part of its offering. It's noted for being the most complete package with features like go karting included along with other benefits such as free socks and special perks for the birthday child."

In [21]:
response = query_engine.query("List all the prices for all the party types")

2026-04-24 18:44:58,570 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/embed "HTTP/1.1 200 OK"
2026-04-24 18:45:01,593 - INFO - HTTP Request: POST https://aerollm.share.zrok.io/api/chat "HTTP/1.1 200 OK"


In [25]:
response.response

'The available birthday party packages at AeroSports Scarborough are priced as follows:\n\n1. **Premium Party**: $39.90 per jumper with a minimum of 10 jumpers, totaling a base price of $399.\n\n2. **VIP Party**: Not explicitly detailed in terms of features and pricing within the provided context, but it is implied to be priced between Premium and Ultimate at $44.90 per jumper.\n\n3. **Ultimate Party**: $59.90 per jumper with a minimum requirement of 10 jumpers, leading to a base price of $599.\n\nAll packages require a minimum of 10 participants and include various amenities such as pizza, drinks, a private party room, a glow t-shirt for the birthday child, digital invitations, and a free 60-minute add-on pass.'